# Simple data center: fixed-load evaluation with DC economics
Same fixed HPP, 16 MW facility load and technical dispatch as simple_datacenter.ipynb.
Costs use installed IT capacity = requested facility load / PUE. Revenue uses
actually served IT energy, including partial supply, valued using the shared YAML.

Facility and IT CAPEX are annualized separately over their economic lifetimes.
Net annual value = DC revenue - annualized DC cost. No HPP costs or PPA payments
are included in this reported value. PPA is excluded from dispatch. DC economics are reporting only for this fixed-load case.

Select hydesign_clean and run all cells. This notebook runs independently.

The EMS uses one CPLEX dispatch solve. Set battery_power_fluctuation_penalty
in YAML (default 0.0; optional trials 0.0001 or 0.001).
Its units are MWh_IT per MW of adjacent change in AC net battery power
(discharge - charge); there is no end-to-start ramp or throughput penalty.
Fixed/basic dispatch maximizes served IT MWh minus this energy-equivalent penalty.
Economic sizing multiplies the penalty by the IT revenue rate to obtain EUR.
Economic reports exclude the artificial penalty; solver objectives/bounds include it.

The unchanged validator rejects simultaneous charging and discharging. The existing
physical formulation permits this, and neither zero nor a positive ramp penalty
guarantees its elimination. The annual reference evaluation can therefore stop with
"Simultaneous battery charging/discharging"; do not use rejected dispatch as a result.


In [ ]:
import os
os.environ['OPENMDAO_REPORTS'] = '0'
import time
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hydesign

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'hydesign/ems/ems_simple_datacenter.py').is_file()), None)
if repo is None or Path(hydesign.__file__).resolve().parent != (repo / 'hydesign').resolve():
    raise RuntimeError('Select the kernel with this HyDesign checkout installed editable.')
print('Python:', sys.executable)
print('HyDesign:', hydesign.__file__)

from hydesign.assembly.hpp_assembly_simple_datacenter_with_dc_cost import hpp_model_simple_datacenter_with_dc_cost as hpp_model
from hydesign.examples import examples_filepath
from hydesign.metrics_simple_datacenter import firm_load_metrics


## Site and simulation parameters
The shared YAML supplies weather-year, battery and solver assumptions.
The explicit reliability_target=0.0 below disables the minimum reliability constraint
for this evaluation only; the YAML reliability target remains available to the sizing notebook.


In [ ]:
examples_sites = pd.read_csv(f'{examples_filepath}examples_sites.csv', index_col=0, sep=';')
name = 'Denmark_good_solar'
ex_site = examples_sites.loc[examples_sites.name == name].iloc[0]
latitude = ex_site.latitude
longitude = ex_site.longitude
altitude = ex_site.altitude
input_ts_fn = examples_filepath + ex_site.input_ts_fn
sim_pars_fn = examples_filepath + 'thesis_simple_datacenter/simple_datacenter.yml'

facility_load_mw = 16.0
PUE = 1.15
PPA = 70.0 # Reporting only; unused by dispatch/sizing objectives.
hpp = hpp_model(
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    work_dir='./',
    sim_pars_fn=sim_pars_fn,
    input_ts_fn=input_ts_fn,
    facility_load_mw=facility_load_mw,
    pue=PUE,
    ppa_price=PPA,
    reliability_target=0.0, # Evaluate achieved reliability without a minimum constraint.
)
print('Site:', name)
print('Weather:', input_ts_fn)
print('Facility demand [MW]:', facility_load_mw)
print('IT demand [MW]:', facility_load_mw / PUE)
print('Hourly weather:', hpp.weather.index[0], 'to', hpp.weather.index[-1], 'N_time =', hpp.N_time)


## Fixed design and evaluation
Use the original HyDesign design variable names and order. PV solar_MW is AC capacity.
Battery b_E = b_P * b_E_h is nominal energy (200 MWh); DoD gives 180 MWh usable.
The reference battery convention gives bus limits of b_P/eta charging and b_P*eta discharging.
Initial and terminal SOC are both 100 MWh. Efficiency in each direction is sqrt(0.86).
This fixed 16 MW evaluation has no minimum reliability constraint; achieved reliability is reported afterwards.


In [ ]:
start = time.time()
clearance = 35
sp = 300
p_rated = 5
Nwt = 20
wind_MW_per_km2 = 7
solar_MW = 180
surface_tilt = 42
surface_azimuth = 180
solar_DCAC = 1.25
b_P = 25
b_E_h = 8

x = [clearance, sp, p_rated, Nwt, wind_MW_per_km2,
     solar_MW, surface_tilt, surface_azimuth, solar_DCAC,
     b_P, b_E_h]
outs = hpp.evaluate(*x)
hpp.print_design(x, outs)
print('Execution time [min]:', (time.time() - start) / 60)
print('Solver status:', hpp.solver_status)


## Hourly results and reliability
The original variable names remain available: ems.hpp_t is served FACILITY power,
ems.hpp_curt_t is curtailment, ems.b_t is net battery discharge, ems.b_E_SOC_t is SOC.
Named hourly arrays are also exposed in hpp.hourly. This evaluation reports achieved reliability without enforcing a minimum.
Its served-energy-optimal dispatch does not necessarily maximize the number of fully supplied hours.


In [ ]:
P_supplied = hpp.prob.get_val('ems.hpp_t').copy()
unserved_facility_mw = hpp.prob.get_val('ems.unserved_facility_mw').copy()
soc = hpp.prob.get_val('ems.b_E_SOC_t').copy()
metrics = firm_load_metrics(unserved_facility_mw, epsilon_mw=hpp.sim_pars['epsilon_mw'])
print(f"LOLH: {metrics['LOLH']:.0f} h/year")
print(f"Successful hours: {metrics['successful_hours']:.0f} / {metrics['total_hours']:.0f}")
print(f"Firm reliability: {metrics['firm_reliability']:.4%}")
print(f"Energy not served: {metrics['EENS']:.3f} MWh/year")

assert len(P_supplied) == 8760 and len(soc) == 8761
np.testing.assert_allclose(P_supplied + unserved_facility_mw, facility_load_mw, atol=1e-5)
assert metrics["LOLH"] == hpp.metrics["LOLH"]
print('Physical validation passed. Maximum residuals:', hpp.validation)
hourly = pd.DataFrame(hpp.hourly, index=hpp.weather.index)
soc_index = hpp.weather.index.append(pd.DatetimeIndex([hpp.weather.index[-1] + pd.Timedelta(hours=1)]))
soc_mwh = pd.Series(soc, index=soc_index, name='soc_mwh')
display(hourly.head())


## Data-center economics
Installed capacity drives all fixed costs. Only served IT energy earns revenue.
The annualized cost component uses the same standalone cost model and YAML assumptions.
Reliability remains a reported result, not an economic adjustment to installed capacity.


In [ ]:
economics = hpp.economics
metrics = hpp.metrics
print(f"Facility load: {economics['facility_load_mw']:.6f} MW")
print(f"Installed IT capacity: {economics['it_capacity_mw']:.6f} MW_IT")
print(f"Served IT energy: {economics['served_it_energy_mwh']:,.3f} MWh_IT/year")
print(f"Served facility energy: {economics['served_facility_energy_mwh']:,.3f} MWh/year")
print(f"Firm IT value: {economics['datacenter_firm_revenue_per_mwh_it']:,.2f} EUR/MWh_IT")
for name in ('CAPEX_dc_facility', 'CAPEX_dc_it', 'CAPEX_dc_total'):
    print(f"{name}: EUR {economics[name]:,.2f}")
for name in ('Revenue_dc', 'OPEX_dc', 'annualized_CAPEX_dc_facility',
             'annualized_CAPEX_dc_it', 'annualized_CAPEX_dc_total',
             'annualized_cost_dc', 'net_annual_value'):
    print(f"{name}: EUR {economics[name]:,.2f}/year")
print(f"Curtailment: {metrics['curtailment_mwh']:,.3f} MWh/year")
print(f"Battery throughput (charge + discharge): {metrics['battery_throughput_mwh']:,.3f} MWh/year")


## Fixed-load dispatch plots
These plots show the evaluated 16 MW case from hpp.hourly and hpp.soc_mwh.


In [ ]:
# Plot a week around the largest shortfall, or the first week if none occurs.
peak = int(np.argmax(hourly.unserved_facility_mw.to_numpy()))
start = max(0, min(peak - 72, len(hourly) - 168))
stop = min(start + 168, len(hourly))
window = hourly.iloc[start:stop]
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(window.index, window.renewable_available_mw, label='Available wind + PV')
axes[0].plot(window.index, window.served_facility_mw, label='Served facility load')
axes[0].plot(window.index, window.facility_load_mw, '--', label='Requested facility load')
axes[0].plot(window.index, window.unserved_facility_mw, label='Unserved facility load')
axes[0].set_ylabel('MW'); axes[0].legend()
axes[1].plot(window.index, window.discharge_mw - window.charge_mw, label='Battery: + discharge / - charge')
axes[1].axhline(0, color='grey', linewidth=0.5)
axes[1].set_ylabel('MW'); axes[1].legend()
axes[2].plot(soc_mwh.iloc[start:stop+1], label='Stored energy')
axes[2].axhline((1-hpp.sim_pars['battery_depth_of_discharge'])*b_P*b_E_h, linestyle='--', color='grey')
axes[2].axhline(b_P*b_E_h, linestyle='--', color='grey')
axes[2].set_ylabel('MWh'); axes[2].legend()
fig.autofmt_xdate(); plt.tight_layout(); plt.show()


## Formulation and limits
The technical constraints are unchanged. One solve maximizes served IT energy
minus the optional net-battery-power fluctuation penalty. The fixed-load run imposes no minimum reliability.

DC economic reporting uses 8760 hourly steps, no degradation, and no external grid.
Revenue_dc = datacenter_firm_revenue_per_mwh_it * sum(served_firm_it_mw).
Installed it_capacity_mw = facility_load_mw / pue, regardless of shortfalls.
The standalone cost model applies separate facility and IT CRFs. The IT lifetime
represents regular replacement through annualization, not an explicit event schedule.
Raw CAPEX is reported separately and is never subtracted directly from annual revenue.
This is DC-only annual value, not whole-project profit: HPP economics are excluded.
